# A simple RAG from scratch

This notebook builds the smallest complete retrieval-augmented generation (RAG) system: a list of facts, an embedding model, an in-memory "vector database", cosine similarity for retrieval, and a language model that answers only from what was retrieved.

It is adapted from ngxson's [demo_simple_rag_py](https://huggingface.co/ngxson/demo_simple_rag_py) (MIT license) and the accompanying post [Code a simple RAG from scratch](https://huggingface.co/blog/ngxson/make-your-own-rag) (October 2024). The original runs two local models through Ollama on a file of cat facts. This version keeps the same steps and data, adds commentary, fixes a few rough edges (trailing newlines and blank lines kept as chunks, a division by zero for empty vectors, an f-string that fails before Python 3.12), and lets you choose the backend:

| `BACKEND` | Embeddings | Generation | Needs |
| :-- | :-- | :-- | :-- |
| `"offline"` (default) | A toy word-hashing embedder | A stand-in that shows the prompt and quotes the best chunk | Nothing; runs anywhere |
| `"azure"` | An Azure OpenAI embedding deployment | An Azure OpenAI chat deployment | Environment variables below |
| `"ollama"` | `bge-base-en-v1.5` (as in the original) | `Llama-3.2-1B-Instruct` (as in the original) | A local Ollama server with both models pulled |

The offline mode is not a real embedding model, it only matches shared words, but it lets you run and inspect every step before spending tokens.



In [1]:
import math, os, re, hashlib, urllib.request

BACKEND = "offline"   # "offline", "azure" or "ollama"
TOP_N = 3

## 1. Load the data and chunk it

The knowledge base is a text file of about 150 short facts about cats, one fact per line. Each line becomes one **chunk**, the unit that is embedded, retrieved and shown to the model. One line per chunk works here because every line is a self-contained fact.

The file is downloaded from the original repository. If there is no network, a few facts are used instead so the notebook still runs.

In [2]:
URL = "https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt"
try:
    raw = urllib.request.urlopen(URL, timeout=15).read().decode("utf-8")
except Exception as e:
    print("Download failed, using a small built-in sample:", e)
    raw = "\n".join([
        "On average, cats spend 2/3 of every day sleeping.",
        "A cat can jump up to six times its length.",
        "Cats have 32 muscles that control the outer ear.",
        "A cat's nose print is unique, much like a human's fingerprint.",
        "The world's largest cat measured 48.5 inches long.",
    ])

# strip newlines and drop empty lines (the original kept both as chunks)
dataset = [line.strip() for line in raw.splitlines() if line.strip()]
print(f"Loaded {len(dataset)} chunks. First three:")
for line in dataset[:3]:
    print(" -", line)

Loaded 150 chunks. First three:
 - On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
 - Unlike dogs, cats do not have a sweet tooth. Scientists believe this is due to a mutation in a key taste receptor.
 - When a cat chases its prey, it keeps its head level. Dogs and humans bob their heads up and down.


## 2. Choose the models

An **embedding model** turns a text into a vector, a list of numbers, so that texts with similar meaning get nearby vectors ([Lesson 03](../03-embedding-models.md)). The same model must embed the chunks and the questions, or their vectors are not comparable. A **language model** writes the answer.

The three backends implement the same two functions, `embed(text)` and `generate(system, question)`, so the rest of the notebook does not change.

In [3]:
if BACKEND == "azure":
    from openai import AzureOpenAI
    client = AzureOpenAI(azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
                         api_key=os.environ["AZURE_OPENAI_API_KEY"],
                         api_version=os.environ.get("OPENAI_API_VERSION", "2024-10-21"))

    def embed(text):
        return client.embeddings.create(model=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"], input=text).data[0].embedding

    def generate(system, question):
        stream = client.chat.completions.create(model=os.environ["AZURE_OPENAI_DEPLOYMENT"], stream=True,
                                                messages=[{"role": "system", "content": system},
                                                          {"role": "user", "content": question}])
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                print(chunk.choices[0].delta.content, end="", flush=True)
        print()

elif BACKEND == "ollama":
    import ollama   # pip install ollama; ollama pull the two models below
    EMBEDDING_MODEL = "hf.co/CompendiumLabs/bge-base-en-v1.5-gguf"
    LANGUAGE_MODEL = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"

    def embed(text):
        return ollama.embed(model=EMBEDDING_MODEL, input=text)["embeddings"][0]

    def generate(system, question):
        for chunk in ollama.chat(model=LANGUAGE_MODEL, stream=True,
                                 messages=[{"role": "system", "content": system},
                                           {"role": "user", "content": question}]):
            print(chunk["message"]["content"], end="", flush=True)
        print()

else:   # offline stand-ins, for studying the pipeline without any model
    DIM = 512
    STOP = {"a", "an", "the", "of", "to", "and", "is", "are", "in", "on", "can", "do", "does", "how", "what", "their", "its", "cat", "cats"}

    def embed(text):
        "Toy embedding: hash each word into one of 512 buckets. Matches shared words, not meaning."
        v = [0.0] * DIM
        for w in re.findall(r"[a-z]+", text.lower()):
            if w not in STOP:
                v[int(hashlib.md5(w.rstrip("s").encode()).hexdigest(), 16) % DIM] += 1.0
        return v

    def generate(system, question):
        print("[offline stand-in: a real model would read this prompt and answer]")
        print(system)
        print("\nQuestion:", question)

## 3. Build the "vector database"

Indexing embeds every chunk once and stores the pair (chunk, vector). Here the store is a plain Python list, which is exactly what a vector database is at its core; real systems add an approximate index so that search does not compare against every vector, plus metadata filters, updates and persistence.

In [4]:
VECTOR_DB = []
for i, chunk in enumerate(dataset):
    VECTOR_DB.append((chunk, embed(chunk)))
print(f"Indexed {len(VECTOR_DB)} chunks, vectors of dimension {len(VECTOR_DB[0][1])}")

Indexed 150 chunks, vectors of dimension 512


## 4. Retrieval: cosine similarity and top-k

**Cosine similarity** measures how closely two vectors point in the same direction: the dot product divided by the product of their lengths, 1 for identical directions and 0 for unrelated ones. Retrieval embeds the question, scores it against every stored vector and keeps the `TOP_N` best. This is exact (flat) search: fine for 150 chunks, too slow for millions, which is why vector databases use approximate indexes.

In [5]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))
    return dot / norm if norm else 0.0      # guard against empty vectors


def retrieve(query, top_n=TOP_N):
    q = embed(query)
    scored = [(chunk, cosine_similarity(q, vec)) for chunk, vec in VECTOR_DB]
    return sorted(scored, key=lambda x: x[1], reverse=True)[:top_n]

## 5. Generation: answer only from the retrieved chunks

The retrieved chunks go into the system message with an instruction to use only that context, which is what makes the answer **grounded**. Printing the retrieved chunks and their scores before the answer is good practice: when an answer is wrong, the first question is whether the right chunk was retrieved at all.

In [6]:
def ask(question):
    hits = retrieve(question)
    print("Retrieved knowledge:")
    for chunk, score in hits:
        print(f" - (similarity {score:.2f}) {chunk}")
    context = "\n".join(f" - {chunk}" for chunk, _ in hits)
    system = ("You are a helpful chatbot. Use only the following pieces of context to answer the question. "
              "Don't make up any new information:\n" + context)
    print("\nChatbot response:")
    generate(system, question)


ask("How much do cats sleep?")

Retrieved knowledge:
 - (similarity 0.53) One reason that kittens sleep so much is because a growth hormone is released only during sleep.
 - (similarity 0.32) Isaac Newton invented the cat flap. Newton was experimenting in a pitch-black room. Spithead, one of his cats, kept opening the door and wrecking his experiment. The cat flap kept both Newton and Spithead happy.
 - (similarity 0.29) The technical term for a cat’s hairball is a “bezoar.”

Chatbot response:
[offline stand-in: a real model would read this prompt and answer]
You are a helpful chatbot. Use only the following pieces of context to answer the question. Don't make up any new information:
 - One reason that kittens sleep so much is because a growth hormone is released only during sleep.
 - Isaac Newton invented the cat flap. Newton was experimenting in a pitch-black room. Spithead, one of his cats, kept opening the door and wrecking his experiment. The cat flap kept both Newton and Spithead happy.
 - The technical term fo

**Reading the result.** In the offline mode, the best fact for this question ("On average, cats spend 2/3 of every day sleeping") is usually not in the top three: the toy embedder matches words, and "sleep" and "sleeping" hash differently, so a fact about kittens' sleep hormone wins. This is failure point 2 of Lesson 00 (the answer exists but is not retrieved), and it is the reason real systems use embedding models that capture meaning. Switch `BACKEND` to `"azure"` or `"ollama"` and compare the retrieved facts.

Try a second question, and one the facts do not cover. With a real model, the instruction to use only the context should lead it to say it does not know; whether it does is exactly what lecture 4 measures as groundedness.

In [9]:
ask("How high can a cat climb?")

Retrieved knowledge:
 - (similarity 0.29) Cats spend nearly 1/3 of their waking hours cleaning themselves.
 - (similarity 0.21) A cat can jump even seven times as high as it is tall.
 - (similarity 0.19) The tiniest cat on record is Mr. Pebbles, a 2-year-old cat that weighed 3 lbs (1.3 k) and was 6.1 inches (15.5 cm) high.

Chatbot response:
[offline stand-in: a real model would read this prompt and answer]
You are a helpful chatbot. Use only the following pieces of context to answer the question. Don't make up any new information:
 - Cats spend nearly 1/3 of their waking hours cleaning themselves.
 - A cat can jump even seven times as high as it is tall.
 - The tiniest cat on record is Mr. Pebbles, a 2-year-old cat that weighed 3 lbs (1.3 k) and was 6.1 inches (15.5 cm) high.

Question: How high can a cat climb?


In [8]:
ask("What is the capital of France?")

Retrieved knowledge:
 - (similarity 0.35) A cat is pregnant for about 58-65 days.
 - (similarity 0.29) The technical term for a cat’s hairball is a “bezoar.”
 - (similarity 0.26) The first cat in space was a French cat named Felicette (a.k.a. “Astrocat”) In 1963, France blasted the cat into outer space. Electrodes implanted in her brains sent neurological signals back to Earth. She survived the trip.

Chatbot response:
[offline stand-in: a real model would read this prompt and answer]
You are a helpful chatbot. Use only the following pieces of context to answer the question. Don't make up any new information:
 - A cat is pregnant for about 58-65 days.
 - The technical term for a cat’s hairball is a “bezoar.”
 - The first cat in space was a French cat named Felicette (a.k.a. “Astrocat”) In 1963, France blasted the cat into outer space. Electrodes implanted in her brains sent neurological signals back to Earth. She survived the trip.

Question: What is the capital of France?


Retrieval always returns `TOP_N` chunks, even when none is relevant: for the question about France it returns the best-scoring facts anyway, such as a fact that mentions France. Nothing in this pipeline says "no good match", so the defence has to come from a similarity threshold, a reranker that can reject candidates, or a model that abstains when the context does not answer the question.